# M1-YOLO Aggressive Re-train (Colab T4)

**목표**: mAP50 **0.85~0.92 도전** (단일 학습으로 0.9 도전)

**전략 (yolov11m + 1280 + 100ep + 강한 augmentation):**
- 모델 yolov11m (v8보다 +3~5 mAP)
- imgsz 1280 (소형 객체 직격타)
- 100 epochs, patience 30
- lr0 5e-4 cosine, freeze 0
- mosaic 1.0, mixup 0.15, copy_paste 0.5

**Drive 준비물:**
1. `structural_compressed.zip` (~6-8GB)

**예상 시간:** 9~12시간 (T4 batch=4, patience=30 종료 시 ~9h)

**Resume + Drive 자동 저장 포함** — 끊겨도 손실 0

**런타임:** 런타임 → 런타임 유형 변경 → GPU → T4

## 1. GPU 확인

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

## 2. 패키지 설치

In [ ]:
!pip install -q ultralytics onnx onnxslim onnxruntime-gpu

## 3. Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 4. 데이터셋 압축 해제 + 자동 저장 폴더 준비

In [ ]:
import os, shutil, zipfile
from pathlib import Path

# ★ Drive에 structural_compressed.zip 올린 폴더 ★
DRIVE_DIR = Path('/content/drive/MyDrive/drone_inspect')
ZIP_NAME = 'structural_compressed.zip'

# 끊김 대비 Drive 자동 저장 폴더
AUTOSAVE = DRIVE_DIR / 'm1_aggressive_autosave'
AUTOSAVE.mkdir(parents=True, exist_ok=True)

WORK = Path('/content/m1_train')
WORK.mkdir(parents=True, exist_ok=True)

zip_path = DRIVE_DIR / ZIP_NAME
print(f'zip exists? {zip_path.exists()} -> {zip_path}')
assert zip_path.exists(), f'{zip_path} 없음'

ds_dir = WORK / 'structural_compressed'
needs_extract = (not ds_dir.exists()) or (not (ds_dir / 'images' / 'val').exists())
if needs_extract:
    if ds_dir.exists():
        shutil.rmtree(ds_dir, ignore_errors=True)
    print('Extracting structural_compressed.zip ...')
    with zipfile.ZipFile(zip_path) as z:
        for m in z.namelist():
            rel = m.replace('\\', '/')
            target = WORK / rel
            if rel.endswith('/'):
                target.mkdir(parents=True, exist_ok=True)
            else:
                target.parent.mkdir(parents=True, exist_ok=True)
                with z.open(m) as src, open(target, 'wb') as dst:
                    shutil.copyfileobj(src, dst)
    print(f'Done: {ds_dir}')
else:
    print(f'Already extracted: {ds_dir}')

for split in ['train', 'val', 'test']:
    p = ds_dir / 'images' / split
    if p.exists():
        cnt = len([f for f in p.glob('*') if f.is_file()])
        print(f'  {split}: {cnt} images')
    else:
        print(f'  {split}: NOT FOUND')

print(f'\nAutosave dir: {AUTOSAVE}')

## 5. data.yaml 생성

In [ ]:
yaml_text = f"""path: {ds_dir}
train: images/train
val: images/val
test: images/test

nc: 3
names:
  0: crack
  1: waterproof_defect
  2: caulking_defect
"""
data_yaml = WORK / 'structural.yaml'
data_yaml.write_text(yaml_text)
print(data_yaml.read_text())

## 6. Resume 가능한지 체크

In [ ]:
# 끊김 후 재실행 시: AUTOSAVE에 last.pt 있으면 거기서 resume
PROJECT = Path('/content/runs/m1_aggressive')
PROJECT.mkdir(parents=True, exist_ok=True)

resume_pt = AUTOSAVE / 'last.pt'
RESUME = False
if resume_pt.exists():
    print(f'Found previous last.pt in Drive: {resume_pt} ({resume_pt.stat().st_size/1024/1024:.1f}MB)')
    print('  → Resume mode')
    # /content/runs/m1_aggressive/train/weights/last.pt로 복원
    weights_dir = PROJECT / 'train' / 'weights'
    weights_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(resume_pt, weights_dir / 'last.pt')
    if (AUTOSAVE / 'best.pt').exists():
        shutil.copy2(AUTOSAVE / 'best.pt', weights_dir / 'best.pt')
    RESUME = True
else:
    print('No previous checkpoint. Fresh aggressive train.')

## 7. 학습 (yolov11m + 1280 + 100ep + Drive 자동 저장)

In [ ]:
from ultralytics import YOLO
import threading, time

# ── Drive 자동 저장 백그라운드 스레드 ──
stop_flag = [False]
def autosave_loop():
    while not stop_flag[0]:
        time.sleep(300)  # 5분마다
        for src in [PROJECT/'train/weights/last.pt', PROJECT/'train/weights/best.pt']:
            if src.exists():
                try:
                    shutil.copy2(src, AUTOSAVE / src.name)
                except Exception as e:
                    print(f'autosave fail: {e}')
        # results.csv도
        rcsv = PROJECT/'train/results.csv'
        if rcsv.exists():
            try: shutil.copy2(rcsv, AUTOSAVE / 'results.csv')
            except: pass
        print(f'[autosave] {time.strftime("%H:%M:%S")} -> {AUTOSAVE}')

t = threading.Thread(target=autosave_loop, daemon=True)
t.start()

# 학습
if RESUME:
    model = YOLO(str(PROJECT / 'train/weights/last.pt'))
    train_kwargs = {'resume': True}
else:
    model = YOLO('yolo11m.pt')  # yolov11m
    train_kwargs = {
        'data': str(data_yaml),
        'epochs': 100,
        'batch': 4,              # T4 16GB + imgsz=1280 → batch=4 안전
        'imgsz': 1280,           # 핵심: 1280 (소형 객체)
        'cache': 'disk',
        'workers': 4,
        'optimizer': 'AdamW',
        'lr0': 5e-4,             # fresh-aggressive
        'lrf': 0.01,             # cosine
        'cos_lr': True,
        'patience': 30,
        'warmup_epochs': 3,
        'close_mosaic': 30,
        'freeze': 0,             # 전체 unfreeze
        'hsv_h': 0.015, 'hsv_s': 0.5, 'hsv_v': 0.4,
        'degrees': 5.0, 'translate': 0.1, 'scale': 0.5,
        'shear': 2.0, 'perspective': 0.001,
        'flipud': 0.0, 'fliplr': 0.5,
        'mosaic': 1.0, 'mixup': 0.15, 'copy_paste': 0.5,  # 강한 augmentation
        'erasing': 0.0,
        'multi_scale': 0.3,
        'save_period': 5,
        'plots': True,
        'project': str(PROJECT.parent),
        'name': PROJECT.name,
        'exist_ok': True,
    }

results = model.train(**train_kwargs)

stop_flag[0] = True
print('Train done.')

## 8. ONNX Export + 평가

In [ ]:
best_path = PROJECT / 'train/weights/best.pt'
print(f'best.pt: {best_path} ({best_path.stat().st_size/1024/1024:.1f}MB)')

best_model = YOLO(str(best_path))
best_model.export(format='onnx', opset=17, dynamic=True, simplify=True)
onnx_path = best_path.with_suffix('.onnx')
print(f'ONNX: {onnx_path} ({onnx_path.stat().st_size/1024/1024:.1f}MB)')

# 평가
metrics = best_model.val(data=str(data_yaml), imgsz=1280, batch=4)
print(f'\n=== M1-Aggressive 결과 ===')
print(f'mAP50:    {metrics.box.map50:.4f}')
print(f'mAP50-95: {metrics.box.map:.4f}')
print(f'precision: {metrics.box.mp:.4f}')
print(f'recall:    {metrics.box.mr:.4f}')
print(f'\n0.9 도달? {"YES ✅" if metrics.box.map50 >= 0.9 else "NO"}')

## 9. 결과를 Google Drive에 저장

In [ ]:
OUT = DRIVE_DIR / 'm1_aggressive_results'
OUT.mkdir(parents=True, exist_ok=True)

shutil.copy2(onnx_path, OUT / 'm1_yolo_structural.onnx')
shutil.copy2(best_path, OUT / 'm1_aggressive_best.pt')

rcsv = best_path.parent.parent / 'results.csv'
if rcsv.exists():
    shutil.copy2(rcsv, OUT / 'results.csv')

for plot in best_path.parent.parent.glob('*.png'):
    shutil.copy2(plot, OUT / plot.name)

print('Saved to:', OUT)
for f in OUT.iterdir():
    print(f'  {f.name} ({f.stat().st_size/1024/1024:.1f}MB)')

## 10. 끝 — 다음 단계 (로컬 PC)

1. Drive `m1_aggressive_results/m1_yolo_structural.onnx` 다운로드
2. 로컬 `backend/models_weights/m1_yolo_structural.onnx`에 덮어쓰기
3. mAP50 확인 (results.csv 마지막 줄 또는 위 평가 출력)